# Generating Tables & Figures

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from plotnine import *
from scipy.stats import spearmanr
import statsmodels.api as sm

from src.constants import (METHOD_ORDER, CUSTOM_MODEL_COLORS, REALDATA_COEXPR_CORRECTEDGES,
                           MIA_METHOD_LABELS, FS_TOKEN_LABELS) 
from src.loaders import combine_result_dfs, combine_mia_dfs
from src.tables import (save_table_html, summarize_metrics_transposed)

from src.plots import save_fig, fig_path


In [ ]:

home_dir = Path.cwd().parent #Path("~/Health-Privacy-Challenge")
dataset_name = "TCGA-BRCA"

FIG_DIR = home_dir / "paper" / "figures" 
MASTER_RES_DIR = home_dir / "results"
PAPER_RES_DIR = home_dir / "paper" / "results"

FIG_DIR.mkdir(parents=True, exist_ok=True)
PAPER_RES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
if not (PAPER_RES_DIR / f"{dataset_name}_util_fidel_mia_results.csv").exists():
    combined_perf_df = combine_result_dfs(home_dir, dataset_name, res_dir="files")
    combined_mia_df = combine_mia_dfs(home_dir, dataset_name, res_dir="mia")
    combined_mia_df_up = combined_mia_df[~((combined_mia_df["method"] == "gan_leaks_original") | 
                                            (combined_mia_df["method"] == "gan_leaks_modified"))]
    
    merged_syn_df = combined_mia_df_up.merge(combined_perf_df, on=["origin", "fold"])

    merged_syn_df.to_csv(PAPER_RES_DIR / f"{dataset_name}_util_fidel_mia_results.csv", index=False)
else:
    merged_syn_df = pd.read_csv(PAPER_RES_DIR / f"{dataset_name}_util_fidel_mia_results.csv")

## Tables 1, 2, 3: Utility, Fidelity, Pathway Recovery

In [ ]:
# Table 1. Fidelity metrics across TCGA-BRCA and TCGA-COMBINED dataset
metrics_df = merged_syn_df.copy() 
metrics_df["is_DP"] = metrics_df["origin"].str.contains("ε=", na=False)

FIDELITY = {
    "MMD_test":                     "MMD (test) ↓",
    "kl_mean_test":                 "KL (test) ↓",
    "discriminative_score":         "Discr. score ↓",
    "distance_to_closest_relative": "DCR (rel.) →1",
    #"prdc_precision":               "Precision",
    #"prdc_recall":                  "Recall",
    "prdc_density_test":             "Density (test) →1",
    "prdc_coverage_test":            "Coverage (test) ↑",
}


## metrics x models 
fidel_t_tbl = summarize_metrics_transposed(
    {dataset_name: metrics_df},
    metrics=list(FIDELITY), method_order=METHOD_ORDER, metric_labels=FIDELITY,
    highlight=True, flat=True,
    lower_is_better=["MMD_test", "kl_mean_test", "discriminative_score"],
    skip_highlight=["prdc_density_test", "distance_to_closest_relative"])
save_table_html(fidel_t_tbl, FIG_DIR / dataset_name / f"{dataset_name}_table1_fidelity.html",
                title="Fidelity", escape=False)

In [ ]:
# Table 2. Utility metrics  across TCGA-BRCA and TCGA-COMBINED datasets
UTILITY = {
    "f1_relative":                "F1 (rel.) ↑",
    "auroc_relative":             "AUROC (rel.)↑",
    #"avg_pr_macro_relative":      "AUPR (rel.)",  
    "feature_overlap_proportion": "Feature overlap ↑",
}
 
## metrics x models 
util_t_tbl = summarize_metrics_transposed(
    {dataset_name: metrics_df},
    metrics=list(UTILITY), method_order=METHOD_ORDER, metric_labels=UTILITY,
    highlight=True, flat=True,
    lower_is_better=["MMD_test", "kl_mean_test", "discriminative_score_relative"],
    skip_highlight=["prdc_density", "distance_to_closest_relative"])
save_table_html(util_t_tbl, FIG_DIR / dataset_name / f"{dataset_name}_table2_utility.html",
                title="Utility", escape=False)

In [ ]:
### Table 3. Pathway recovery performance across TCGA-BRCA and TCGA-COMBINED datasets.
from src.loaders import combine_pathway_dfs
from src.stats import prepare_pathway_summary
from src.plots import plot_pathway_heatmap

PATHWAY_METRIC_LABELS = {"ks_stat": "Mean KS ↓", "abs_effect": "Mean |effect| ↓",
                         "var_ratio_geom": "Var ratio →1", "frac_sig_fdr": "Frac sig (FDR) ↓"}

pw_df = combine_pathway_dfs(home_dir, dataset_name)
perfold, per_pathway = prepare_pathway_summary(pw_df, alpha=0.05)

## metrics x models 
pathway_t_tbl = summarize_metrics_transposed(
    {dataset_name: perfold},
    metrics=list(PATHWAY_METRIC_LABELS), 
    method_order=METHOD_ORDER, metric_labels=PATHWAY_METRIC_LABELS,
    highlight=True, flat=True,
    lower_is_better=["ks_stat", "abs_effect", "frac_sig_fdr"],
    skip_highlight=["var_ratio_geom"])
save_table_html(pathway_t_tbl, FIG_DIR / dataset_name / f"{dataset_name}_table3_pathway_fidelity.html",
                title="Pathway", escape=False)

In [ ]:
### Fig. S7-S8: Pathway recovery in the TCGA-BRCA dataset, shown as signed effect sizes (Cohen's d) 
# between synthetic and real ssGSEA scores across generative models and MSigDB Hallmark gene sets.
fig = plot_pathway_heatmap(per_pathway, dataset_name=dataset_name,
                           value_col="effect_size", method_order=METHOD_ORDER)
save_fig(fig, fig_path(FIG_DIR, dataset_name, "pathway_effect"), size=(8, 12))

## Main and Supplementary Figures

In [ ]:
# Fig. 2: Differential Expression (DE) recovery analysis for TCGA-BRCA and TCGA-COMBINED datasets.
LFC_THRESHOLD = 0.5
FPR_MAX       = 0.05

# %% --- DE gene recovery ---
from src.loaders import combine_bio_dfs, prepare_de_points
from src.plots import plot_de_recovery

if not (PAPER_RES_DIR / f"{dataset_name}_DE={LFC_THRESHOLD}_results.csv").exists():
    combined_de_df = combine_bio_dfs(home_dir, dataset_name, LFC_THRESHOLD)
    combined_de_df.to_csv(PAPER_RES_DIR / f"{dataset_name}_DE={LFC_THRESHOLD}_results.csv", index=False)
else:
    combined_de_df = pd.read_csv(PAPER_RES_DIR / f"{dataset_name}_DE={LFC_THRESHOLD}_results.csv")

de_points, de_folds = prepare_de_points(combined_de_df, FPR_MAX)

de_main_fig = plot_de_recovery(
    de_points, dataset_name=dataset_name,
    lfc_threshold=LFC_THRESHOLD, fpr_max=FPR_MAX,
    colors=CUSTOM_MODEL_COLORS, dp_split=False, dp_emphasis=True,    # <- the clean DP/non-DP separation
)
save_fig(de_main_fig, fig_path(FIG_DIR, dataset_name, "DE", lfc=LFC_THRESHOLD, fpr=FPR_MAX), # 
        size=(9, 6))

In [ ]:
### Fig. S4-S5: Differential expression (DE) recovery difficulty across pairs
from src.loaders import prepare_comparison_difficulty
from src.plots import plot_comparison_difficulty

diff = prepare_comparison_difficulty(combined_de_df)
fig_difficulty = plot_comparison_difficulty(diff, dataset_name=dataset_name,
                                 lfc_threshold=LFC_THRESHOLD, fpr_max=FPR_MAX,
                                 colors=CUSTOM_MODEL_COLORS)
diff_size = (8, 8)
if "COMBINED" in dataset_name: diff_size = (11, 8)
save_fig(fig_difficulty, fig_path(FIG_DIR, dataset_name, "DE_difficulty", lfc=LFC_THRESHOLD),
        size=diff_size)


In [ ]:
# Fig. S6: Sensitivity of differential-expression (DE) recovery to the effect-size threshold ΔVST

from src.loaders import  prepare_de_lfc_sweep
from src.plots import plot_de_lfc_sweep

lfc_sweep = prepare_de_lfc_sweep(
    home_dir, dataset_name, lfc_thresholds=[0.0, 0.2, 0.5, 1.0], fpr_max=FPR_MAX,
    combine_fn=combine_bio_dfs, prepare_fn=prepare_de_points,
    all_models=METHOD_ORDER, average_direction=True)      # <- new flag

fig = plot_de_lfc_sweep(lfc_sweep, dataset_name=dataset_name, fpr_max=FPR_MAX,
                        colors=CUSTOM_MODEL_COLORS)
save_fig(fig, fig_path(FIG_DIR, dataset_name, "DE_lfc_sensitivity"), size=(12, 5))

In [ ]:
# Fig. 3: Co-expression recovery analysis for the TCGA-BRCA and TCGA-COMBINED datasets. 

COEXPR_THRESHOLDS = [0.0, 0.3, 0.5]  

# %% --- co-expression edge recovery ---
from src.loaders import combine_coexpr_dfs, prepare_coexpr_points
from src.plots import plot_coexpr_recovery


if not (PAPER_RES_DIR / f"{dataset_name}_coexpr_results.csv").exists():
    combined_coexpr = combine_coexpr_dfs(
        home_dir, dataset_name, COEXPR_THRESHOLDS,
        real_edges=REALDATA_COEXPR_CORRECTEDGES[dataset_name],
    )
    combined_coexpr.to_csv(PAPER_RES_DIR / f"{dataset_name}_coexpr_results.csv", index=False)
else:
    combined_coexpr = pd.read_csv(PAPER_RES_DIR / f"{dataset_name}_coexpr_results.csv")


coexpr_summary, coexpr_folds = prepare_coexpr_points(combined_coexpr)
coexpr_fig = plot_coexpr_recovery(coexpr_summary, dataset_name=dataset_name,
                           colors=CUSTOM_MODEL_COLORS, dp_split=False, dp_emphasis=True, log=False)   # or False
save_fig(coexpr_fig, fig_path(FIG_DIR, dataset_name, "CoE",
                       thr="-".join(map(str, COEXPR_THRESHOLDS))),
        size=(9, 5))

## Membership inference attack

In [ ]:
from src.loaders import prepare_mia_full
from src.tables import format_mia_full_table, save_table_html
from src.constants import FS_TOKENS 


## removing MC because of unstability from best-performing attack plots in main text
privacy_tables = merged_syn_df[~(merged_syn_df["method"]=="MC")].copy()
## keeping MC for supplementary figures
privacy_for_heatmap = merged_syn_df.copy()

full = prepare_mia_full(privacy_tables, metrics=("tpr_at_fpr_01", "mia_aucroc"),
                        fs_tokens=FS_TOKENS)
mia_full_tbl = format_mia_full_table(full, method_order=METHOD_ORDER)
save_table_html(mia_full_tbl, f"../results/figures/{dataset_name}/{dataset_name}_table_MIA_full.html",
                title="MIA attack surface (by attack × feature selection)")

In [ ]:
# Fig. 4. Membership inference attack (MIA) performance on the TCGA-BRCA and TCGA-COMBINED datasets.
from src.loaders import prepare_mia_worstcase_folds
from src.plots import plot_mia_worstcase_box

box_size = (6, 7)
if "COMBINED" in dataset_name: box_size = (6, 8)
worst_folds = prepare_mia_worstcase_folds(privacy_tables,
                                          metrics=("tpr_at_fpr_01", "mia_aucroc"),
                                          fs_tokens=FS_TOKENS)
mia_box_fig = plot_mia_worstcase_box(worst_folds, dataset_name=dataset_name, annotate="shape")
save_fig(mia_box_fig, fig_path(FIG_DIR, dataset_name, "MIA_worstcase_box"), size=box_size)

In [ ]:
# Supplementary figures S9-S14: MIA heatmaps for TCGA-BRCA and TCGA-COMBINED datasets.
from src.loaders import prepare_mia_heatmap
from src.plots import plot_mia_heatmap

for metric, label, floor in [("tpr_at_fpr_01", "TPR@FPR=0.1", 0.1),
                             ("mia_aucroc", "AUC-ROC", 0.5),
                             ("mia_pr_auc", "AUC-PR", 0.8)]:
    heat = prepare_mia_heatmap(privacy_for_heatmap, metric, fs_tokens=FS_TOKENS,
                               attack_labels=MIA_METHOD_LABELS, fs_labels=FS_TOKEN_LABELS)
    heat_fig = plot_mia_heatmap(heat, dataset_name=dataset_name, metric_label=label,
                           method_order=METHOD_ORDER, floor=floor)
    save_fig(heat_fig, fig_path(FIG_DIR, dataset_name, "MIA_heatmap", metric=metric),
              size=(6, 4))

In [ ]:
# Fig. S15-S16: Impact of feature selection on membership-inference risk 
# across the three reference-anchored attacks (GAN-leaks calibrated, DOMIAS, LOGAN),
# Fig. S17: Agreement between feature-selection variants in 
# how they rank generators by membership-inference risk, for the TCGA-COMBINED dataset only. 
from src.loaders import prepare_fs_leakage, fs_rank_agreement_multi
from src.plots import plot_fs_leakage_heatmap, plot_fs_rank_agreement, save_fig, fig_path
from src.constants import FS_TOKENS, FS_TOKEN_LABELS, METHOD_ORDER

CAPTION_FS = ("Spearman ρ of generator leakage rankings between feature-selection "
             "variants, under both MIA metrics.\nPooled as worst-case (max) over the "
              "three reference-anchored methods (gan_leaks_cal, LOGAN, DOMIAS-KDE); "
              "n=11 generators.\nHigh ρ indicates that feature selection does not reorder which "
               "generators are vulnerable.")
metric_lbl_dict = {"mia_aucroc": "AUC-ROC", "tpr_at_fpr_01": "TPR@FPR=0.1"}

if "COMBINED" in dataset_name:
    for metric, floor in [("mia_aucroc", 0.5), ("tpr_at_fpr_01", 0.1)]:
        pooled, per = prepare_fs_leakage(privacy_tables, metric=metric,
            fs_tokens=FS_TOKENS, main_methods=("gan_leaks_cal","LOGAN_D1","domias_kde"),
            collapse="max")
        fig_a = plot_fs_leakage_heatmap(per, dataset_name=dataset_name,
            metric_label=metric_lbl_dict[metric],
            method_order=METHOD_ORDER, fs_order=["none","pca","vDE","sDE","dDE"],
            fs_labels=FS_TOKEN_LABELS, floor=floor)
        save_fig(fig_a, fig_path(FIG_DIR, dataset_name, f"fs_leakage_{metric}"), size=(12, 5))

    long = fs_rank_agreement_multi(privacy_tables,
        metrics=["mia_aucroc", "tpr_at_fpr_01"], fs_tokens=FS_TOKENS,
        main_methods=("gan_leaks_cal","LOGAN_D1","domias_kde"), collapse="max",
        fs_order=["none","pca","vDE","sDE","dDE"],
        metric_labels={"mia_aucroc":"AUC-ROC", "tpr_at_fpr_01":"TPR@FPR=0.1"})
    fig_c = plot_fs_rank_agreement(long, dataset_name=dataset_name,
        fs_labels=FS_TOKEN_LABELS, fs_order=["none","pca","vDE","sDE","dDE"],
        caption=CAPTION_FS)
    save_fig(fig_c, fig_path(FIG_DIR, dataset_name, "fs_rank_agreement"), size=(10, 5))


In [ ]:
# Fig. S18: Spearman correlations between privacy measures across generative models, per MIA attack. 
from src.stats import correlations_per_attack
from src.plots import save_fig, fig_path, plot_rho_per_attack

privacy_metric_corr = privacy_for_heatmap.copy() 

pairs = [("tpr_at_fpr_01","mia_aucroc"), ("distance_to_closest","tpr_at_fpr_01"),
         ("distance_to_closest","mia_aucroc"), ("tpr_at_fpr_01","tpr_at_fpr_001"),]

PAIR_LABELS = {
    "tpr_at_fpr_01 vs mia_aucroc":          "TPR@FPR=0.1 vs AUC-ROC",
    "distance_to_closest vs tpr_at_fpr_01": "DCR vs TPR@FPR=0.1",
    "distance_to_closest vs mia_aucroc":    "DCR vs AUC-ROC",
    "tpr_at_fpr_01 vs tpr_at_fpr_001":      "TPR@FPR=0.1 vs TPR@FPR=0.01",
}

rho = correlations_per_attack(privacy_metric_corr, pairs, fs_tokens=FS_TOKENS,
                              target_metrics=())
fig_rho = plot_rho_per_attack(rho, dataset_name=dataset_name, pair_labels=PAIR_LABELS,
                          sort_by="tpr_at_fpr_01 vs mia_aucroc")
save_fig(fig_rho, fig_path(FIG_DIR, dataset_name, "rho_per_attack"), size=(6, 8))

## Correlation heatmaps

In [ ]:
# Fig. S19-S20: Spearman correlation heatmap of selected metric from four evaluation axes: 
# fidelity, downstream utility, biological plausibility and privacy. 

from src.constants import CORR_METRICS, CORR_TARGET_METRICS, CORR_METRIC_LABELS
from src.stats import compute_metric_correlations
from src.plots import plot_correlation_heatmap, save_fig, fig_path
from src.loaders import assemble_per_method_metrics

filtered_df = merged_syn_df.copy() #[mask]
## removing MC because of unstability
tradeoffs_df = filtered_df[~(filtered_df["method"]=="MC")]

UF_MODEL_METRICS = ("MMD_test", "MMD_train", "kl_mean_test", "kl_mean_train",
                    "discriminative_score_relative", "prdc_density_test", "prdc_coverage_test",
                    "f1_relative", "avg_pr_macro_relative", "distance_to_closest_relative")

per_method = assemble_per_method_metrics(
    tradeoffs_df, combined_coexpr, combined_de_df, pw_df,
    coexpr_threshold=0.3, fpr_metric="mia_aucroc")
# using AUC-ROC for MIA because it is more stable than TPR@FPR=0.1 across folds, 
# change to tpr_at_fpr_01 if you want to see the correlation with TPR@FPR=0.1 instead. 
# watch the log: it warns if any method isn't in all four sources (n would shrink). 

corr = compute_metric_correlations(per_method, CORR_METRICS,
                                   target_metrics=CORR_TARGET_METRICS)
n_corr = corr['n']

CAPTION_SEL = (f"Spearman ρ across {n_corr} generative models.\n"
           f"Benjamini/Hochberg FDR, α = 0.05, q = adjusted p\n" 
           f"*: q < 0.05; **:  q < 0.01; ***: q < 0.001\n" 
           f"DE threshold: ΔVST=0.5; co-expression threshold r > 0.3\n ") 

fig_sel = plot_correlation_heatmap(corr, dataset_name=dataset_name,cluster=True,
                               metric_labels=CORR_METRIC_LABELS, caption=CAPTION_SEL)
save_fig(fig_sel, fig_path(FIG_DIR, dataset_name, "sel_metric_corr"), size=(9, 8))

In [ ]:

# Fig. S2-S3: Spearman correlation heatmap of complete fidelity and downstream utility metrics. 
UF_MODEL_METRICS = ("MMD_test", "MMD_train", "kl_mean_test", "kl_mean_train", "feature_overlap_proportion",
                    "discriminative_score", "prdc_density", "prdc_coverage",
                     "prdc_density_test", "prdc_coverage_test",
                    "f1_relative", "avg_pr_macro_relative", "distance_to_closest_relative")

per_method_uf = assemble_per_method_metrics(
    tradeoffs_df, combined_coexpr, combined_de_df, pw_df,
    coexpr_threshold=0.3, fpr_metric="mia_aucroc",
    model_metrics=UF_MODEL_METRICS)        

UF = ["coexpr_tpr", "de_tpr"] + list(UF_MODEL_METRICS)

corr_uf = compute_metric_correlations(per_method_uf, UF_MODEL_METRICS,
              target_metrics=("prdc_density", "distance_to_closest_relative"))

n_corr = corr_uf['n']
CAPTION_UTIL = (f"Spearman ρ across {n_corr} generative models.\n"
               f"Benjamini/Hochberg FDR, α = 0.05, q = adjusted p\n" 
               f"*: q < 0.05; **:  q < 0.01; ***: q < 0.001" ) 
fig = plot_correlation_heatmap(corr_uf, dataset_name=dataset_name,
          metric_labels=CORR_METRIC_LABELS, cluster=True,
          caption=CAPTION_UTIL)
save_fig(fig, fig_path(FIG_DIR, dataset_name, "corr_utilfid_clustered"), size=(9, 8))

## Trade-off plots

In [ ]:
#  Fig. 5. Membership inference risk across evaluation axes
from src.loaders import prepare_tradeoff
from src.plots import plot_tradeoff

LOWER_BETTER = {"privacy_tpr", "MMD_test", "kl_mean_test",
                "discriminative_score", "pathway_ks"}

CAPTION_PRIV = (
           f"Benjamini/Hochberg FDR, α = 0.05, q = adjusted p\n" 
           f"*: q < 0.05; **:  q < 0.01; ***: q < 0.001\n" 
           f"DE threshold: ΔVST=0.5; co-expression threshold r > 0.3\n ") #TPR@FPR=0.1.


to_priv = prepare_tradeoff(per_method, corr["long"], anchor="privacy_tpr",
    against=("coexpr_tpr", "de_tpr", "f1_relative", "MMD_test"),
    lower_is_better=LOWER_BETTER, metric_labels=CORR_METRIC_LABELS)

save_fig(plot_tradeoff(to_priv, dataset_name=dataset_name, colors=CUSTOM_MODEL_COLORS,
    ncol=4, privacy_floor=0.5,
    anchor_label="MIA (highest AUC-ROC)", #TPR@FPR
    title=f"{dataset_name} — Membership Inference Risk across Evaluation Axes", caption=CAPTION_PRIV, 
    point_size=7.5),
    fig_path(FIG_DIR, dataset_name, "tradeoff_privacy"), size=(10, 5))


In [ ]:
# Fig. 6. DE recovery across evaluation axes. 
CAPTION_DE = (
           f"Benjamini/Hochberg FDR, α = 0.05, q = adjusted p\n" 
           f"*: q < 0.05; **:  q < 0.01; ***: q < 0.001\n" 
           f"DE threshold: ΔVST=0.5; co-expression threshold r > 0.3\n ") #TPR@FPR=0.1.

to_de = prepare_tradeoff(per_method, corr["long"], anchor="de_tpr",
    against=("f1_relative", "discriminative_score", "pathway_ks", "coexpr_tpr"),
    lower_is_better=LOWER_BETTER, metric_labels=CORR_METRIC_LABELS)
save_fig(plot_tradeoff(to_de, dataset_name=dataset_name, colors=CUSTOM_MODEL_COLORS,
    ncol=4, anchor_label="DE recovery TPR",
    title=f"{dataset_name} — DE recovery across Evaluation Axes", caption=CAPTION_DE, point_size=7.5),
    fig_path(FIG_DIR, dataset_name, "tradeoff_de"), size=(10, 5))

In [ ]:
# Fig. 7. MMD (test) across evaluation axes. 
CAPTION_MMD = (
           f"Benjamini/Hochberg FDR, α = 0.05, q = adjusted p\n" 
           f"*: q < 0.05; **:  q < 0.01; ***: q < 0.001\n" 
           f"Co-expression threshold r > 0.3\n ") #TPR@FPR=0.1.

to_mmd = prepare_tradeoff(per_method, corr["long"], anchor="MMD_test",
    against=("de_tpr", "coexpr_tpr", "pathway_ks", "f1_relative"),
    lower_is_better=LOWER_BETTER, metric_labels=CORR_METRIC_LABELS)
save_fig(plot_tradeoff(to_mmd, dataset_name=dataset_name, colors=CUSTOM_MODEL_COLORS,
    ncol=4, anchor_label="MMD (test)",
    title=f"{dataset_name} — Fidelity (MMD) across Evaluation Axes", caption=CAPTION_MMD, point_size=7.5),
    fig_path(FIG_DIR, dataset_name, "tradeoff_mmd"), size=(10, 5))


## Ablations

In [ ]:
# Supplementary figures S1, S21-S22: Ablation of ε (privacy budget) for DP models, 
# and noise level for MVN models, and P-NMF across evaluation axes.
from src.loaders import assemble_per_method_metrics, prepare_epsilon_ablation
from src.plots import plot_epsilon_ablation
from src.constants import (DPMODEL_PATHS, DPMODEL_COLORS, 
                           PNMF_MODEL_PATHS, PNMF_MODEL_COLORS,
                          REALDATA_COEXPR_CORRECTEDGES, 
                          MVN_MODEL_PATHS, MVN_MODEL_COLORS)

ablation_tests = {"DP": [DPMODEL_PATHS, DPMODEL_COLORS, "ε"], 
                  "MVN": [MVN_MODEL_PATHS, MVN_MODEL_COLORS, "noise"], 
                  "P-NMF": [PNMF_MODEL_PATHS, PNMF_MODEL_COLORS, "ε"]}

for k, v in ablation_tests.items():
    model_paths = v[0]
    model_colors = v[1]
    epsilon_label = v[2]
    # load all four sources for the DP models 
    res_dp = combine_result_dfs(home_dir, dataset_name, model_paths=model_paths)
    mia_dp = combine_mia_dfs(home_dir, dataset_name, model_paths=model_paths)
    merged_dp = res_dp.merge(mia_dp, on=["origin", "fold"])

    coexpr_dp = combine_coexpr_dfs(home_dir, dataset_name, COEXPR_THRESHOLDS, REALDATA_COEXPR_CORRECTEDGES[dataset_name],
                                model_paths=model_paths   )
    de_dp     = combine_bio_dfs(home_dir, dataset_name, LFC_THRESHOLD, model_paths=model_paths)
    pw_dp     = combine_pathway_dfs(home_dir, dataset_name, model_paths=model_paths)

    per_method_dp = assemble_per_method_metrics(merged_dp, coexpr_dp, de_dp, pw_dp,
                                                coexpr_threshold=0.0,
                                                fpr_metric="mia_aucroc") #tpr_at_fpr_01

    abl_util = prepare_epsilon_ablation(per_method_dp, x_metric="f1_relative",
                                        y_metric="privacy_tpr", from_per_method=True)
    abl_util["panel"] = "Utility (F1 rel.)"
    abl_de = prepare_epsilon_ablation(per_method_dp, x_metric="de_tpr",
                                    y_metric="privacy_tpr", from_per_method=True)
    abl_de["panel"] = "DE recovery (TPR)"
    abl = pd.concat([abl_util, abl_de], ignore_index=True)

    fig = plot_epsilon_ablation(abl, dataset_name=dataset_name, colors=model_colors,
            point_size=7, facet_col="panel", ncol=2, connect=False,
            x_label="Quality metric", y_label="Privacy (highest MIA AUC-ROC)",
            privacy_floor=0.5,
            title=f"{dataset_name} — {k} vs quality across {epsilon_label}")
    save_fig(fig, fig_path(FIG_DIR, dataset_name, f"epsilon_ablation_{k}"), size=(7, 4))